# Multivariate Matrix Profile Motif Discovery Visual Story

This notebook shows what changes when repeated subsequences must agree jointly across several financial variables. It uses the repository feature pipeline and `run_multivariate_matrix_profile`, preserving the historical STUMPY MSTUMP behavior.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while not (ROOT / "reports" / "final_story" / "scripts" / "final_story_core.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Could not locate thesis repository root.")
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "reports" / "final_story" / "scripts"))
from final_story_core import *

ensure_dirs()
configure_plots()
print(f"Repository root: {ROOT}")
print(f"Data source: {DATA_PATH}")
print(f"Selected period: {PERIOD_START} to {PERIOD_END}")

## 1. Multivariate Representation

The visual example uses a compact feature subset drawn from available and historically used features: return, realized volatility, high-low range, and volume activity. The historical full-feature benchmark remains referenced for runtime comparison.

In [ ]:
multi = build_multivariate_context()
write_run_record('multivariate')
multi['diagnostics']

In [ ]:
display(multivariate_feature_panel(multi))
pd.read_csv(MULTI_TAB / '01_multivariate_feature_selection.csv')

## 2. Scaled Representation

MSTUMP receives robust-scaled features from the existing feature-selection methodology. This panel is not the raw market representation.

In [ ]:
display(multivariate_scaled_panel(multi))

## 3. Multivariate Matrix Profile

The historical implementation computes `stumpy.mstump(matrix.T, window_length)` and selects `dimension_row = min(n_features - 1, profile_matrix_rows - 1)`. For four selected features, this corresponds to the row requiring agreement across the full selected dimensionality after MSTUMP's subspace ordering.

In [ ]:
display(multivariate_profile_context(multi))
pd.read_csv(MULTI_TAB / '03_multivariate_selected_motif_metadata.csv')

## 4. Strongest Multivariate Motif Across Channels

The motif pair is shown feature-by-feature using the same robust-scaled values used for discovery.

In [ ]:
display(multivariate_top_across_features(multi))
pd.read_csv(MULTI_TAB / '04_multivariate_top_motif_feature_values.csv').head()

## 5. Univariate Versus Multivariate Motif

The two methods are compared at the same window length and period. Distances are reported but not interpreted on a common scale.

In [ ]:
display(univariate_vs_multivariate(multi))
pd.read_csv(MULTI_TAB / '05_univariate_vs_multivariate_top_motif_comparison.csv')

## 6. Case Study Where Additional Dimensions Matter

The selection rule searches the top univariate candidates for a pair with low univariate distance but relatively larger disagreement in at least one other scaled feature channel.

In [ ]:
display(multivariate_information_case(multi))
pd.read_csv(MULTI_TAB / '06_when_multivariate_information_changes_similarity_case_candidates.csv').head()

## 7. Multivariate Top-5 Motif Gallery

Each panel summarizes a top multivariate pair using the mean of selected scaled channels for compact visual comparison.

In [ ]:
display(multivariate_gallery(multi))
pd.read_csv(MULTI_TAB / '07_multivariate_top5_motifs.csv')

## 8. Similarity Structure

This auxiliary matrix uses RMS aggregation of per-feature z-normalized Euclidean distances. It is not the MSTUMP profile itself.

In [ ]:
display(multivariate_similarity(multi))
pd.read_csv(MULTI_TAB / '08_multivariate_top5_similarity_matrix.csv', index_col=0)

## 9. Feature-Set Comparison

The saved historical benchmark is used here. Its central message is computational: the full multivariate representation costs much more runtime than simple univariate feature sets. This does not imply universal motif-quality improvement.

In [ ]:
display(feature_set_comparison())
pd.read_csv(MULTI_TAB / '09_feature_set_comparison.csv')

## 10. Feature-Number / Dimensionality Consideration

STUMPY MSTUMP returns rows associated with increasing subspace dimensionality. The figure below records the minimum profile value by row for the selected compact feature set and documents the row semantics used by the historical pipeline.

In [ ]:
display(dimensionality_profile(multi))
pd.read_csv(MULTI_TAB / '10_multivariate_dimensionality_profile.csv')

## 11. Concise Findings

The executed evidence supports limited descriptive claims: multivariate motifs can require coherence across return, volatility, range, and volume activity; the strongest multivariate pair may differ from the univariate pair; adding dimensions changes the geometry of subsequence similarity; and historical feature-set benchmarks show a major runtime cost for broad multivariate representations. Nonstationarity and regimes are examined later in the thesis.

In [ ]:
validation = validate_outputs('multivariate')
manifest = write_manifest()
print('Manifest:', manifest)
print('Output figures:')
for p in sorted(MULTI_FIG.glob('*.png')):
    print(p)
print('Output CSVs:')
for p in sorted(MULTI_TAB.glob('*.csv')):
    print(p)
validation